# Names

Date: 2024/01/21, 2024/04/19(SQLite), 2025/07/12(Gemini replacing spaCy)

In [27]:
#!pip3 install google-genai
#!pip3 install pandas

In [28]:

import google.genai as genai
import os

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
MODEL = "gemini-2.5-flash"

client = genai.Client(api_key=GEMINI_API_KEY)

## Preprocessing and Paragraph-Level Splitting of the Original Text

In [29]:
import sqlite3

import pandas as pd
with sqlite3.connect('../data/bach.db') as conn:
    paragraphs = pd.read_sql('SELECT * FROM paragraphs', conn)
    
paragraphs.head()

,index,paragraph
0,0,If there is such a thing as inherited aptitude...
1,1,"Veit Bach, ancestor of this famous family, gai..."
2,2,"Not all the Bachs, however, were great musicia..."
3,3,We do not know whether they rewarded the expec...
4,4,"Besides these three men, the Bachs boasted sev..."


In [30]:
len(paragraphs)

159

In [31]:
response = client.models.generate_content(
    model=MODEL,
    contents=f"""
    Extract the names of all persons mentioned in the following paragraphs.
    If a name is mentioned multiple times, include it only once.
    If a name is misspelled, correct it.
    If a name is not full name, make it full name.

    ## Paragraphs     
    {paragraphs.paragraph.to_list()}
    """,
    config={
        "response_mime_type": "application/json",
        "response_schema": list[str],
    }
)

NAMES = response.parsed
NAMES

['Altnikol, Johann Christoph',
 'Augustus II of Poland-Saxony',
 'Augustus III of Poland-Saxony',
 'Bach, Anna Magdalena',
 'Bach, Carl Philipp Emmanuel',
 'Bach, Johann Ambrosius',
 'Bach, Johann Christian',
 'Bach, Johann Christoph (Arnstadt)',
 'Bach, Johann Christoph (Ohrdruf)',
 'Bach, Johann Christoph Friedrich',
 'Bach, Johann Sebastian',
 'Bach, Regine Susanna',
 'Bach, Veit',
 'Bach, Wilhelm Friedemann',
 'Benda',
 'Berardi, Angelo',
 'Beethoven, Ludwig van',
 'Birnbaum, Johann Abraham',
 'Böhm, Georg',
 'Bononcini, Giovanni Battista',
 'Bruhns, Nicolaus',
 'Buxtehude, Dietrich',
 'Caldara, Antonio',
 'Charles III of Spain',
 'Couperin, François',
 'Duke Christian of Weissenfels',
 'Duke Ernst August of Weimar',
 'Elias Gottlieb Haussmann',
 'Erdmann',
 'Faustina Bordoni Hasse',
 'Fischer, Johann Caspar Ferdinand',
 'Flemming, Marshal Count',
 'Frescobaldi, Girolamo',
 'Froberger, Johann Jakob',
 'Fux, Johann Joseph',
 'Gesner, Johann Matthias',
 'Goldberg',
 'Görner, Johann G

## Export data

In [32]:
import sqlite3
import pandas as pd

names = pd.Series(NAMES, name='name')
with sqlite3.connect('../data/bach.db') as conn:
    names.to_sql('names', conn, if_exists='replace')    